In [ ]:
pip install pycaret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of category-encoders to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of pmdarima to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of pyod to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.9/58.9 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 3.1 MB/s eta 0:00:00


In [15]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. Montar o Google Drive
drive.mount('/content/drive')

# 2. Carregar os dados (ajuste o caminho da pasta se necessário)
caminho_arquivo = '/content/drive/MyDrive/TI-Saude/Atividade Zamberlan/dados_saude_predicao.csv'

try:
    df = pd.read_csv(caminho_arquivo)
    print("Arquivo carregado com sucesso!\n")
except FileNotFoundError:
    print(f"Erro: O arquivo não foi encontrado no caminho: {caminho_arquivo}")
    print("Verifique se o nome do arquivo ou a estrutura de pastas no seu Google Drive correspondem.")

# 3. Features e variável alvo para o problema de saúde
features = ['Idade', 'Pressao_Arterial', 'Colesterol_Total', 'Frequencia_Cardiaca_Max']
target = 'Risco_Internacao'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Arquivo carregado com sucesso!



In [16]:
from pycaret.classification import ClassificationExperiment
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import pandas as pd # Ensure pandas is imported for DataFrame operations

In [17]:
# 3. Inicializar e configurar o experimento do PyCaret
exp = ClassificationExperiment()
exp.setup(
    data=df,
    target=target,
    numeric_features=features,
    train_size=0.7,
    session_id=42,
    normalize=True,
    normalize_method='zscore',
    html=False,
    verbose=False,
    fix_imbalance=True # Adicionado para lidar com o desbalanceamento das classes
)

print("PyCaret Setup Complete!")

PyCaret Setup Complete!


In [18]:
# 4. Obter conjuntos de treino e teste processados pelo PyCaret
X_train = exp.get_config('X_train')
y_train = exp.get_config('y_train')
X_test = exp.get_config('X_test')
y_test = exp.get_config('y_test')

print("Train and Test sets retrieved from PyCaret.")

Train and Test sets retrieved from PyCaret.


In [20]:
# 5. Criar e avaliar os modelos especificados no PyCaret
model_ids = {
    'Logistic Regression': 'lr',
    'Decision Tree': 'dt',
    'Random Forest': 'rf',
    'KNN': 'knn',
    'Naive Bayes': 'nb',
    'SVM': 'rbfsvm',
    'Gradient Boosting': 'gbc'
}

resultados = []

print("=" * 60)
print("DETALHAMENTO DOS MODELOS (PYCARET)")
print("=" * 60)

for nome, model_id in model_ids.items():
    # Criar modelo sem saída verbosa durante a criação
    modelo = exp.create_model(model_id, verbose=False)

    # Fazer previsões diretamente nos X_train e X_test pré-processados do PyCaret
    y_pred_train = modelo.predict(X_train)
    y_pred_test = modelo.predict(X_test)

    # Calcular métricas
    acc_train = accuracy_score(y_train, y_pred_train)
    f1_train = f1_score(y_train, y_pred_train, average='macro', zero_division=0)

    acc_test = accuracy_score(y_test, y_pred_test)
    f1_test = f1_score(y_test, y_pred_test, average='macro', zero_division=0)

    resultados.append({
        'Modelo': nome,
        'Acc Treino': acc_train,
        'F1 Treino': f1_train,
        'Acc Teste': acc_test,
        'F1 Teste': f1_test
    })

    print(f"\nModelo: {nome}")
    print(f"Acurácia (Treino): {acc_train:.4f} | Acurácia (Teste): {acc_test:.4f}")
    print(f"F1-Score Macro (Treino): {f1_train:.4f} | F1-Score Macro (Teste): {f1_test:.4f}")
    print("\nMatriz de Confusão (Teste):")
    print(confusion_matrix(y_test, y_pred_test))
    print("\nRelatório de Classificação (Teste):")
    print(classification_report(y_test, y_pred_test, zero_division=0))
    print("-" * 60)


DETALHAMENTO DOS MODELOS (PYCARET)

Modelo: Logistic Regression
Acurácia (Treino): 0.5029 | Acurácia (Teste): 0.4933
F1-Score Macro (Treino): 0.3346 | F1-Score Macro (Teste): 0.3304

Matriz de Confusão (Teste):
[[ 0 38]
 [ 0 37]]

Relatório de Classificação (Teste):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        38
           1       0.49      1.00      0.66        37

    accuracy                           0.49        75
   macro avg       0.25      0.50      0.33        75
weighted avg       0.24      0.49      0.33        75

------------------------------------------------------------

Modelo: Decision Tree
Acurácia (Treino): 0.5029 | Acurácia (Teste): 0.4933
F1-Score Macro (Treino): 0.3346 | F1-Score Macro (Teste): 0.3304

Matriz de Confusão (Teste):
[[ 0 38]
 [ 0 37]]

Relatório de Classificação (Teste):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        38
          

In [ ]:
# 6. Exibir Ranking Final em Formato de Tabela
df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values(by='F1 Teste', ascending=False)

print("\n" + "=" * 60)
print("RANKING FINAL DOS MODELOS (Ordenado por F1-Score no Teste)")
print("=" * 60)
print(df_resultados.to_string(index=False, formatters={
    'Acc Treino': '{:.4f}'.format,
    'F1 Treino': '{:.4f}'.format,
    'Acc Teste': '{:.4f}'.format,
    'F1 Teste': '{:.4f}'.format
}))


RANKING FINAL DOS MODELOS (Ordenado por F1-Score no Teste)
             Modelo Acc Treino F1 Treino Acc Teste F1 Teste
        Naive Bayes     0.4971    0.3321    0.5067   0.3363
Logistic Regression     0.5029    0.3346    0.4933   0.3304
      Decision Tree     0.5029    0.3346    0.4933   0.3304
      Random Forest     0.5029    0.3346    0.4933   0.3304
                KNN     0.5029    0.3346    0.4933   0.3304
                SVM     0.5029    0.3346    0.4933   0.3304
  Gradient Boosting     0.5029    0.3346    0.4933   0.3304
